# 14 - Executive Assistant
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/14-exec-assistant/exec_assistant_workbook.ipynb)

Turn an email thread or meeting transcript into a draft reply, action items with owners and deadlines, and a follow-up tracker -- all in a single typed LLM call.

**Harness focus:** Fan-out output -- one input produces multiple typed schemas simultaneously

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: The fan-out pattern -- one input, three typed outputs

Most agent patterns produce one output from one input. The fan-out pattern produces multiple structured outputs from a single LLM call.

Instead of calling the LLM three times (reply, action items, follow-ups), we define one Pydantic model that contains all three as sub-structures. This is more efficient and ensures the outputs are consistent with each other.

```
Email thread
    |
    v
[One LLM call]
    |
    v
ExecOutput
  draft_reply         <-- one typed string
  action_items[]      <-- list of ActionItem
  follow_up_tracker[] <-- list of FollowUpEntry
```

## Part 2: Schema -- one model containing three sub-structures

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel

class ActionItem(BaseModel):
    description: str
    owner: Optional[str] = None
    due_date: Optional[str] = None
    priority: Literal["high", "medium", "low"]

class FollowUpEntry(BaseModel):
    topic: str
    waiting_on: Optional[str] = None
    check_in_by: Optional[str] = None
    notes: str

class ExecOutput(BaseModel):
    input_type: Literal["email_thread", "meeting_transcript"]
    draft_reply: str
    subject_line: Optional[str] = None
    action_items: List[ActionItem]
    follow_up_tracker: List[FollowUpEntry]
    meeting_summary: Optional[str] = None

print("Schema defined.")

## Part 3: System prompt -- enforce all outputs are populated

The key instruction: every output must be populated. If there is nothing substantive to reply to, draft a brief acknowledgement.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

ASSISTANT_SYSTEM = SystemMessage("""
You are a world-class executive assistant.

From the input produce ALL of the following in one pass:
1. draft_reply      -- A polished, ready-to-send reply or acknowledgement.
2. action_items     -- Every discrete task with owner and deadline where identifiable.
3. follow_up_tracker -- Items that need monitoring but are not direct tasks yet.
4. meeting_summary  -- Populate only if the input is a meeting transcript.
5. subject_line     -- Populate only if the input is an email thread.

Set input_type = 'email_thread' or 'meeting_transcript'.
Every output must be populated -- no empty lists.""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
assistant = ASSISTANT_SYSTEM | llm.with_structured_output(ExecOutput)
print("Agent ready.")

## Part 4: Run on an email thread

In [ ]:
EMAIL_THREAD = """
From: Sarah Connor <pm@techcorp.com>
To: James Reed <cto@techcorp.com>; CEO <ceo@techcorp.com>
Subject: RE: API Integration Deadline -- URGENT

James / David, the Vendor X API integration is 15 days past the 1 June deadline.
We have an SLA clause: 5% monthly fee rebate per week of delay.
Legal needs to advise on our exposure. I need a decision now. -- Sarah

---
From: James Reed <cto@techcorp.com>
The integration was deprioritised for the security audit. Poor communication -- I accept.
I can restart by Wed 18 June, deliver by 9 July.
Suggest a call with Vendor X this week. -- James

---
From: David Okafor <ceo@techcorp.com>
Call with both of you Wednesday. Legal: SLA exposure by EOD Tuesday. -- David
"""

result = assistant.invoke(HumanMessage(content="Input to process:\n\n" + EMAIL_THREAD))
print(f"Type: {result.input_type} | Actions: {len(result.action_items)} | Follow-ups: {len(result.follow_up_tracker)}")

## Part 5: Run on a meeting transcript sample

In [ ]:
MEETING_TRANSCRIPT = """
MEETING: Weekly product sync -- 16 June 2025
Attendees: Alice (CPO), Ben (Eng lead), Carol (Design)

Alice: The new onboarding flow needs to ship by end of June. Ben, can your team commit?
Ben: We need the final designs from Carol by Friday or we cannot hit the date.
Carol: I will have designs ready by Thursday EOD. I need copy from marketing -- Alice can you chase?
Alice: Yes, I will email them today. Ben, also please set up the staging environment by Monday.
Ben: Will do. I will also add Carol to the Jira board so she can track progress.
"""

result2 = assistant.invoke(HumanMessage(content="Input to process:\n\n" + MEETING_TRANSCRIPT))
print(f"Type: {result2.input_type}")
print(f"Summary: {result2.meeting_summary}")
print(f"\nAction items ({len(result2.action_items)}):")
for a in result2.action_items:
    owner = a.owner or 'TBC'
    print(f"  [{a.priority.upper()}] {a.description} -- {owner}")

## Exercise: Add a risk_flag field

Add a `risk_flag: Optional[str]` field to `ExecOutput` that the model populates when it detects potential legal or financial exposure in the input (e.g. SLA breach, contract dispute, compliance issue).

Leave it `None` when there is no detectable risk.

**Starter:** Add the field and update the system prompt.

In [ ]:
# Starter: extend ExecOutput with risk_flag
class ExecOutputV2(BaseModel):
    input_type: Literal["email_thread", "meeting_transcript"]
    draft_reply: str
    subject_line: Optional[str] = None
    action_items: List[ActionItem]
    follow_up_tracker: List[FollowUpEntry]
    meeting_summary: Optional[str] = None
    risk_flag: Optional[str] = None  # NEW: describe risk if detected

# TODO: update ASSISTANT_SYSTEM to instruct the model to populate risk_flag
# TODO: rebuild assistant_v2 = updated_system | llm.with_structured_output(ExecOutputV2)
# TODO: run on EMAIL_THREAD and print risk_flag

In [ ]:
# Answer key
ASSISTANT_SYSTEM_V2 = SystemMessage("""
You are a world-class executive assistant.

From the input produce:
1. draft_reply, action_items, follow_up_tracker as before.
2. risk_flag: if you detect legal or financial exposure (SLA breach, contract dispute,
   regulatory issue, financial liability), describe it in one sentence.
   Otherwise set it to null.
Every output must be populated -- no empty lists for action_items or follow_up_tracker.""")

assistant_v2 = ASSISTANT_SYSTEM_V2 | llm.with_structured_output(ExecOutputV2)
result_v2 = assistant_v2.invoke(HumanMessage(content="Input to process:\n\n" + EMAIL_THREAD))
print(f"Risk flag: {result_v2.risk_flag}")
print(f"Actions: {len(result_v2.action_items)}")

## What You Built

- A fan-out agent: one LLM call produces three typed sub-structures
- `ExecOutput` as a single Pydantic model containing `action_items[]`, `follow_up_tracker[]`, and `draft_reply`
- System prompt enforces population of all outputs
- Input type detection (email vs transcript) built into the schema

**Next steps:** Try example 15 (citation-grounded extraction) or wire this fan-out output into an email sending tool.